# Project 01 — Titanic Survival EDA

**Difficulty:** Beginner  
**Skills:** Pandas, Matplotlib, Seaborn, EDA  
**Dataset:** Seaborn's built-in Titanic (no download needed)

## Objective
Explore the Titanic passenger manifest to understand what factors determined survival. Answer key questions:
1. What was the overall survival rate?
2. Did gender affect survival?
3. Did passenger class affect survival?
4. Did age play a role?
5. Did family size matter?
6. What features are most predictive of survival?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Load dataset — no download needed
df = sns.load_dataset('titanic')
print('Shape:', df.shape)
print(df.head())

## 1. Data Overview

In [ ]:
df.info()

In [ ]:
print('Missing values:')
print(df.isna().sum())
print('\nMissing %:')
print((df.isna().mean()*100).round(1))

In [ ]:
print(df.describe().T.round(2))

## 2. Survival Rate Overview

In [ ]:
survived_count = df['survived'].value_counts()
print('Survived:', survived_count[1], f'({survived_count[1]/len(df)*100:.1f}%)')
print('Perished:', survived_count[0], f'({survived_count[0]/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=df, x='survived', palette=['coral','steelblue'], ax=axes[0])
axes[0].set_xticklabels(['Perished', 'Survived'])
axes[0].set_title('Survival Count')
axes[0].set_xlabel('')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height()}\n({p.get_height()/len(df)*100:.1f}%)',
                     (p.get_x()+p.get_width()/2, p.get_height()+5), ha='center')

axes[1].pie(survived_count, labels=['Perished','Survived'],
            autopct='%1.1f%%', colors=['coral','steelblue'], startangle=90,
            explode=(0, 0.05))
axes[1].set_title('Survival Rate')

plt.suptitle('Titanic — Overall Survival', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Survival by Gender

In [ ]:
gender_survival = df.groupby('sex')['survived'].agg(['sum','count','mean'])
gender_survival.columns = ['survived','total','rate']
gender_survival['rate'] = gender_survival['rate'].round(3)
print(gender_survival)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=df, x='sex', hue='survived', palette=['coral','steelblue'], ax=axes[0])
axes[0].set_title('Count by Gender')
axes[0].legend(['Perished','Survived'])
axes[0].set_xlabel('')

sex_rates = df.groupby('sex')['survived'].mean()
sex_rates.plot.bar(color=['coral','steelblue'], ax=axes[1], rot=0, edgecolor='white')
for i, v in enumerate(sex_rates):
    axes[1].text(i, v+0.01, f'{v:.1%}', ha='center', fontweight='bold')
axes[1].set_title('Survival Rate by Gender')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)

plt.suptitle('"Women and Children First" — Gender Effect', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Survival by Passenger Class

In [ ]:
print(df.groupby('pclass')['survived'].agg(['sum','count','mean']).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='pclass', hue='survived', palette=['coral','steelblue'], ax=axes[0])
axes[0].set_title('Count by Passenger Class')
axes[0].legend(['Perished','Survived'])
axes[0].set_xlabel('Passenger Class')

pclass_rates = df.groupby('pclass')['survived'].mean()
pclass_rates.plot.bar(color=['gold','silver','#cd7f32'], ax=axes[1], rot=0, edgecolor='white')
for i, v in enumerate(pclass_rates):
    axes[1].text(i, v+0.01, f'{v:.1%}', ha='center', fontweight='bold')
axes[1].set_title('Survival Rate by Passenger Class')
axes[1].set_xlabel('Passenger Class (1=First, 3=Third)')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)

plt.suptitle('Wealth = Better Survival — Class Effect', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Survival by Age

In [ ]:
# Handle missing ages
df_age = df.dropna(subset=['age'])
print(f'Rows with age: {len(df_age)}/{len(df)} ({len(df_age)/len(df):.1%})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age distribution by survival
sns.histplot(data=df_age, x='age', hue='survived', bins=30, multiple='stack',
             palette=['coral','steelblue'], ax=axes[0])
axes[0].set_title('Age Distribution by Survival')
axes[0].legend(['Perished','Survived'])

# Violin plot
sns.violinplot(data=df_age, x='survived', y='age', palette=['coral','steelblue'], ax=axes[1])
axes[1].set_xticklabels(['Perished','Survived'])
axes[1].set_title('Age Distribution — Survived vs Perished')
axes[1].set_xlabel('')

plt.suptitle('Age and Survival', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Children (<12) survival rate
df['is_child'] = df['age'] < 12
print('\nChild survival rate:', df[df['is_child']]['survived'].mean().round(3))
print('Adult survival rate:', df[~df['is_child']]['survived'].mean().round(3))

## 6. Survival by Family Size

In [ ]:
df['family_size'] = df['sibsp'] + df['parch'] + 1  # +1 = self
df['family_type'] = df['family_size'].apply(
    lambda x: 'Alone' if x == 1 else 'Small (2-4)' if x <= 4 else 'Large (5+)'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

family_rates = df.groupby('family_size')['survived'].mean()
family_rates.plot.bar(color='steelblue', ax=axes[0], rot=0, edgecolor='white', alpha=0.8)
for i, v in enumerate(family_rates):
    axes[0].text(i, v+0.01, f'{v:.0%}', ha='center', fontsize=9)
axes[0].set_title('Survival Rate by Family Size')
axes[0].set_xlabel('Family Size (including self)')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

famtype_rates = df.groupby('family_type')['survived'].mean().reindex(['Alone','Small (2-4)','Large (5+)'])
famtype_rates.plot.barh(color=['coral','steelblue','green'], ax=axes[1], edgecolor='white')
for i, v in enumerate(famtype_rates):
    axes[1].text(v+0.01, i, f'{v:.1%}', va='center', fontweight='bold')
axes[1].set_title('Survival Rate by Family Type')
axes[1].set_xlabel('Survival Rate')
axes[1].set_xlim(0, 1)

plt.suptitle('Family Size and Survival', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Multi-dimensional Analysis

In [ ]:
# Survival by class AND gender
fig, ax = plt.subplots(figsize=(10, 5))
survival_matrix = df.groupby(['pclass','sex'])['survived'].mean().unstack()
survival_matrix.plot.bar(color=['coral','steelblue'], rot=0, edgecolor='white', ax=ax)
ax.set_title('Survival Rate by Class & Gender', fontsize=13)
ax.set_xlabel('Passenger Class')
ax.set_ylabel('Survival Rate')
ax.set_ylim(0, 1.1)
ax.legend(['Female','Male'])
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{bar.get_height():.0%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: Class × Gender survival rate
pivot = df.groupby(['pclass','sex'])['survived'].mean().unstack()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.1%', cmap='RdYlGn', vmin=0, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'format': '%.0%%'})
ax.set_title('Survival Rate: Class × Gender')
ax.set_xlabel('')
ax.set_ylabel('Passenger Class')
plt.tight_layout()
plt.show()

## 8. Fare Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='pclass', y='fare', palette=['gold','silver','#cd7f32'], ax=axes[0])
axes[0].set_title('Fare Distribution by Class')
axes[0].set_xlabel('Passenger Class')
axes[0].set_yscale('log')

sns.scatterplot(data=df.dropna(subset=['age']), x='age', y='fare', hue='survived',
                palette=['coral','steelblue'], alpha=0.5, ax=axes[1])
axes[1].set_title('Age vs Fare (coloured by survival)')
axes[1].legend(['Perished','Survived'])

plt.tight_layout()
plt.show()

## Key Findings

| Factor | Finding |
|--------|--------|
| **Overall** | Only 38.4% of passengers survived |
| **Gender** | Females had 74.2% survival rate vs males at 18.9% |
| **Class** | 1st class: 63% | 2nd class: 47% | 3rd class: 24% |
| **Age** | Children (<12) had ~57% survival rate |
| **Family** | Small families (2-4) survived best (~58%); solo travellers (~30%) |
| **Fare** | Higher fare strongly correlated with higher class and survival |

## Next Steps

Now that we understand the data, we can build a classifier:
- Feature engineering: `family_size`, `title` extracted from name
- Handle missing `age` (impute by median per class+sex)
- Encode categoricals: `sex`, `embarked`
- Train: Logistic Regression, Random Forest, XGBoost
- Target: ~80%+ accuracy is achievable

→ See **Section 05 – Machine Learning** for the modelling step.